# Lab 3b: Sequence Modeling dengan LSTM

**Pendamping Bab 03 (Eksperimen Reproduksibel).**

Tujuan: menambahkan keluarga arsitektur sequence (RNN/LSTM) ke repertoar Anda. Anda akan melatih LSTM untuk *one-step-ahead prediction* pada sinyal sine+noise sintetis, mendemonstrasikan perbedaan gradient flow antara vanilla RNN dan LSTM, dan menggunakan praktik reproducibility dari Bab 03 (seed, config, logging).

**Prasyarat:** Lab 1 (baseline CNN), Lab 1c (MLP numpy), dan Section 2.0b (backprop + vanishing gradient).

**Estimasi:** 3-4 jam.

## 0. Setup

### Lokal
```bash
pip install -e .
```

### Google Colab
Jalankan sel kode **0a. Setup Colab** di bawah ini. Setelah selesai, lanjut ke **1. Setup dan smoke test**.

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/muhammad-zainal-muttaqin/ModulePembelajaran.git'
    REPO_DIR = '/content/ModulePembelajaran'

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    else:
        print('Repo sudah ada, skip clone.')

    %cd {REPO_DIR}/ModulePembelajaran/template_repo
    !pip install -e .
    print('\nSetup Colab selesai. Working dir:', os.getcwd())
else:
    print('Environment lokal. Lewati setup Colab.')

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from src.models import SimpleLSTM

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 2. Dataset sintetis: sine + noise

Kita bangun dataset forecasting sederhana - diberi 50 langkah sebelumnya, prediksi langkah ke-51. Dataset sintetis dipilih karena: (a) tidak perlu download, (b) pola struktur pasti sehingga kita tahu model "yang jujur bekerja" akan konvergen, (c) bisa divariasikan tingkat noise-nya untuk diagnostik.

In [ ]:
def make_sine_dataset(n_samples=4000, seq_len=50, noise=0.05, seed=42):
    rng = np.random.default_rng(seed)
    starts = rng.uniform(0, 2 * np.pi, size=n_samples)
    freqs = rng.uniform(0.5, 2.0, size=n_samples)
    t = np.linspace(0, 4 * np.pi, seq_len + 1)
    data = np.sin(starts[:, None] + freqs[:, None] * t)  # (n, seq_len+1)
    data += rng.normal(0, noise, data.shape)
    X = data[:, :-1][:, :, None].astype(np.float32)      # (n, seq_len, 1)
    y = data[:, -1:].astype(np.float32)                  # (n, 1)
    return X, y

X_tr, y_tr = make_sine_dataset(4000, 50, noise=0.05, seed=42)
X_va, y_va = make_sine_dataset(500, 50, noise=0.05, seed=1337)

print('X train:', X_tr.shape, 'y train:', y_tr.shape)
plt.plot(X_tr[0, :, 0]); plt.scatter([50], y_tr[0], color='red', label='target'); plt.legend(); plt.title('satu sampel')
plt.show()

## 3. Vanilla RNN vs LSTM: gradient flow

Sebelum training, kita ilustrasikan *vanishing gradient* dengan backpropagate gradient dari loss di timestep terakhir ke setiap timestep input, lalu plot norm gradient per-timestep. Untuk RNN dalam, gradient di timestep awal biasanya nyaris nol - itulah kenapa LSTM didesain.

In [ ]:
def gradient_norm_per_timestep(cell_type='RNN', seq_len=50, hidden=32):
    torch.manual_seed(0)
    if cell_type == 'RNN':
        cell = nn.RNN(1, hidden, batch_first=True)
    else:
        cell = nn.LSTM(1, hidden, batch_first=True)
    x = torch.randn(1, seq_len, 1, requires_grad=True)
    out, _ = cell(x)
    loss = out[:, -1, :].sum()
    loss.backward()
    g = x.grad[0, :, 0].abs().numpy()
    return g

g_rnn  = gradient_norm_per_timestep('RNN')
g_lstm = gradient_norm_per_timestep('LSTM')
plt.plot(g_rnn,  label='vanilla RNN', marker='.')
plt.plot(g_lstm, label='LSTM',        marker='.')
plt.yscale('log'); plt.xlabel('timestep (0 paling lama, 49 terbaru)')
plt.ylabel('|gradient| di input'); plt.title('Gradient flow lewat sequence'); plt.legend()
plt.show()
print('Gradient awal RNN:  ', g_rnn[0], '| akhir:', g_rnn[-1])
print('Gradient awal LSTM: ', g_lstm[0], '| akhir:', g_lstm[-1])

## 4. Training loop

Pakai `SimpleLSTM` dari `src/models.py` agar reproducibility (seed, config) konsisten dengan praktik Bab 03.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

model = SimpleLSTM(input_size=1, hidden_size=64, num_layers=1, num_classes=1, readout='last').to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
crit = nn.MSELoss()

ds_tr = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
dl_tr = DataLoader(ds_tr, batch_size=64, shuffle=True)
X_va_t = torch.tensor(X_va).to(device); y_va_t = torch.tensor(y_va).to(device)

hist_tr, hist_va = [], []
for epoch in range(20):
    model.train(); running = 0.0
    for Xb, yb in dl_tr:
        Xb, yb = Xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(model(Xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # selalu clip pada sequence model
        opt.step()
        running += loss.item() * len(Xb)
    model.eval()
    with torch.no_grad():
        va_loss = crit(model(X_va_t), y_va_t).item()
    hist_tr.append(running / len(ds_tr)); hist_va.append(va_loss)
    print(f'epoch {epoch+1:2d}  tr={hist_tr[-1]:.4f}  va={va_loss:.4f}')

In [ ]:
plt.plot(hist_tr, label='train'); plt.plot(hist_va, label='val')
plt.xlabel('epoch'); plt.ylabel('MSE'); plt.legend(); plt.title('LSTM sine forecasting')
plt.show()

## 5. Inspeksi prediksi

Visualisasi: prediksi vs ground truth pada 6 sampel validasi.

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(X_va_t[:6]).cpu().numpy()
fig, ax = plt.subplots(2, 3, figsize=(11, 5))
for i, a in enumerate(ax.ravel()):
    a.plot(X_va[i, :, 0]); a.scatter([50], y_va[i], color='black', label='truth')
    a.scatter([50], pred[i], color='red', marker='x', label='pred')
    a.set_title(f'sampel {i}')
    a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6. Refleksi

1. Pada cell 3, gradient timestep awal (indeks 0) seberapa besar untuk RNN vs LSTM? Apakah hasil Anda konsisten dengan prediksi *vanishing gradient* di Section 2.0b?
2. Coba matikan `clip_grad_norm_` pada training. Apakah loss masih stabil? Di titik mana (kalau terjadi) Anda melihat sinyal *exploding gradient*?
3. Gantikan `SimpleLSTM` dengan `SimpleMLP(input_dim=50, hidden_sizes=(64,), num_classes=1)` (dengan `x.flatten(start_dim=1)`). Apakah MLP bisa mencapai val loss yang sama? Mengapa atau mengapa tidak? Hubungkan jawaban Anda dengan *efisiensi statistik* yang dibahas di akhir Section 2.0b.